# 第7回 CNN：値とコードを一段ずつ追う

画像 → 窓と積和 → 出力サイズ → ReLUと集約 → 分類 → 学習 → 比較、の順に進みます。
前提はPythonのリスト、添字、for文、関数です。`image[行][列]`は0から数えます。`append`は末尾に1個追加、`enumerate`は番号と値を組にします。

**開始：** VS Codeで既存venvのカーネルを選び、最初は変更せず「カーネル再起動→すべて実行」で動作を確かめます。その後、一節ずつ戻って読みます。学習中はセルの実行が終わるまで待ちます。
Notebook単独で動き、同梱.pyのimport、外部データ、通信、GPUは不要です。依存の初回導入だけは別途必要です。

**完了の目印：** 畳み込みの1行目[-2,2,0]、100回後の学習用平均損失約0.130153、学習用4/4・確認用2/4。
表示は小数を丸めますが計算は丸めません。図の色に加えて行・列・太枠・名前を読んでください。

[操作教材](https://nshrhm.github.io/intro-ai/chap06-part01/interactive.html) ／ [スライド](https://nshrhm.github.io/intro-ai/chap06-part01/)（同じ入力・計算条件を図と操作で確認できます）


In [ ]:
import sys
print("使用中のPython:", sys.executable)
print("バージョン:", sys.version.split()[0])


In [ ]:
"""第7回: 自作4×4画像でCNNの計算と小さな学習を観察する。

python cnn_basics.py --demo (--json は検証用)
入力は人工画像だけ。外部データ・GPU・追加ライブラリは不要。
学習率0.1・100更新と、未学習の端の画像の予測を比較する。
数値微分は28パラメーター限定の教育用。大規模学習向けではない。
"""
import json
import math
import random
from copy import deepcopy

# Python Random(42): フィルター8値、バイアス2値、出力重み16値、バイアス2値。
_rng = random.Random(42)
INITIAL_WEIGHTS = tuple([_rng.uniform(-.5, .5) for _ in range(8)] + [.1, .1]
                        + [_rng.uniform(-.5, .5) for _ in range(16)] + [0., 0.])
IMAGE = [[0, 1, 0, 0] for _ in range(4)]
FILTER = [[1, -1], [1, -1]]


In [ ]:
CLASS_NAMES = ('縦線', '横線')


### 図表を表示する準備（計算本体とは別）

以下は表示用の補助関数です。変更せず実行します。HTML/SVGの書き方を覚える必要はありません。折り畳みは表示環境により異なります。


In [ ]:
VIEW_STYLE = '<style>\n.cnn-view {font:16px/1.6 sans-serif;color:#172b42;background:white;padding:12px;max-width:1100px}\n.cnn-view .boards {display:flex;gap:18px;flex-wrap:wrap;align-items:start}\n.cnn-view table {border-collapse:collapse;margin:8px 0 16px;font-size:16px}\n.cnn-view td,.cnn-view th {border:1px solid #7f8d9c;padding:6px 10px;text-align:right}\n.cnn-view caption {text-align:left;font-weight:bold;color:#172b42}\n.cnn-view .pixel {min-width:28px;text-align:center}\n.cnn-view .one {background:#ddeaf5}\n.cnn-view .pad {background:#f6ead2;border:2px dashed #805d25}\n.cnn-view .focus {outline:3px solid #155e75;outline-offset:-3px;font-weight:bold}\n.cnn-view .winner {text-decoration:underline;text-decoration-thickness:3px}\n.cnn-view .scroll {max-width:100%;overflow-x:auto}\n.cnn-view svg {max-width:100%;height:auto}\n.cnn-view .note {border-left:4px solid #155e75;padding-left:12px}\n</style>'


In [ ]:
from html import escape


In [ ]:
from IPython.display import HTML, display


In [ ]:
def number(value):
    return f'{value:.6g}' if isinstance(value, float) else str(value)


In [ ]:
def show(content):
    display(HTML(VIEW_STYLE + '<div class="cnn-view">' + content + '</div>'))


In [ ]:
def table(title, headings, rows):
    header = ''.join('<th>' + escape(str(x)) + '</th>' for x in headings)
    body = ''
    for row in rows:
        body += '<tr>' + ''.join('<td>' + escape(number(x)) + '</td>' for x in row) + '</tr>'
    return '<div class="scroll"><table><caption>' + escape(title) + '</caption><tr>' + header + '</tr>' + body + '</table></div>'


In [ ]:
def grid(title, values, focus=(), padding=0, winners=()):
    headings = '<th>行↓ 列→</th>' + ''.join(f'<th>{j}</th>' for j in range(len(values[0])))
    body = ''
    for i, row in enumerate(values):
        body += f'<tr><th>{i}</th>'
        for j, value in enumerate(row):
            added = padding and (i < padding or j < padding or i >= len(values)-padding or j >= len(row)-padding)
            classes = ['pixel', 'pad' if added else 'one' if value else '']
            if (i, j) in focus:
                classes.append('focus')
            if (i, j) in winners:
                classes.append('winner')
            label = '追加0' if added else number(value)
            body += '<td class="' + ' '.join(classes) + '">' + escape(label) + '</td>'
        body += '</tr>'
    return '<table><caption>' + escape(title) + '</caption><tr>' + headings + '</tr>' + body + '</table>'


In [ ]:
def window_view(image, kernel, stride=1, padding=0, position=0):
    records = []
    output = convolve(image, kernel, stride, padding, record=records.append)
    if not 0 <= position < len(records):
        raise ValueError(f'positionは0〜{len(records)-1}です')
    item = records[position]
    height, width = len(image), len(image[0])
    padded = [[0] * (width+2*padding) for _ in range(height+2*padding)]
    for i, row in enumerate(image):
        for j, value in enumerate(row):
            padded[i+padding][j+padding] = value
    top, left = item['start']
    focus = [(top+i, left+j) for i in range(len(kernel)) for j in range(len(kernel[0]))]
    out_focus = [tuple(item['output'])]
    boards = grid('入力：太枠内が窓／破線は追加0', padded, focus, padding)
    boards += grid('掛ける重み（手作り）', kernel)
    boards += grid('出力：太枠1マスへ置く', output, out_focus)
    rows = [(f"[{t['row']},{t['col']}]", '追加0' if t['added_zero'] else '元画像', t['pixel'], t['weight'], t['product']) for t in item['terms']]
    show('<div class="boards">' + boards + '</div>' + table(f"窓{position+1}/{len(records)} 開始{item['start']} → 出力{item['output']} 合計{item['total']}", ['元画像の位置', '由来', '画素', '重み', '積'], rows))
    return records


In [ ]:
def pooling_view(image, mode, position=0):
    records = []
    output = pool(image, mode, record=records.append)
    if not 0 <= position < len(records):
        raise ValueError(f'positionは0〜{len(records)-1}です')
    item = records[position]
    top, left = item['start']
    focus = [(top+i, left+j) for i in range(2) for j in range(2)]
    winners = [point for point in focus if image[point[0]][point[1]] == max(item['values'])] if mode == 'max' else []
    formula = f"最大{max(item['values'])}（該当する全マスに下線）" if mode == 'max' else f"合計{sum(item['values'])} ÷ 4"
    show('<div class="boards">' + grid('ReLU後：太枠が4値', image, focus, winners=winners) + grid('集約後：太枠が出力', output, [tuple(item['output'])]) + '</div>' + table(formula, ['窓内の4値', '結果'], [(str(item['values']), item['total'])]))


In [ ]:
def bars(probabilities):
    body = ''
    for i, value in enumerate(probabilities):
        y = 30+i*52
        body += f'<text x="8" y="{y}">{CLASS_NAMES[i]}</text><rect x="80" y="{y-18}" width="{value*300}" height="24" fill="#196780"/><text x="395" y="{y}">{value:.6f}</text>'
    return '<svg viewBox="0 0 530 130" role="img" aria-label="縦線と横線のsoftmax出力。棒の全幅は1">' + body + '<text x="80" y="122">0</text><text x="370" y="122">1</text></svg>'


In [ ]:
def classification_view(weights, image, label):
    result = forward(weights, image)
    show(grid('入力画像／正解：' + CLASS_NAMES[label], image))
    for k in range(2):
        kernel = [weights[k*4:k*4+2], weights[k*4+2:k*4+4]]
        boards = grid('重み ' + 'ab'[k] + f'／バイアス={weights[8+k]:.6f}', kernel)
        for key, title in [('maps','畳み込み＋バイアス'),('relu','ReLU'),('pooled','集約')]:
            boards += grid('ab'[k] + '：' + title, result[key][k])
        show('<div class="boards">' + boards + '</div>')
    names = ['a1','a2','a3','a4','b1','b2','b3','b4']
    show(table('各2×2を左上→右上→左下→右下の順に並べる', names, [result['features']]))
    for c in range(2):
        terms = score_terms(weights, result['features'], c)
        rows = [(t['name'],t['feature'],t['weight'],t['product']) for t in terms]
        show(table(CLASS_NAMES[c] + 'のスコア：8個の積の合計＋バイアス', ['名前','特徴量','分類の重み','積'], rows) + f"<p>バイアス {weights[26+c]:.6f} → スコア {result['logits'][c]:.6f}</p>")
    prediction = predict(weights, [(image,label)])[0]
    show(bars(result['probabilities']) + '<p>予測：' + CLASS_NAMES[prediction] + ' ／ 正解：' + CLASS_NAMES[label] + '。大きい出力を予測とする。同値なら縦線。</p>')


In [ ]:
def sample_view(weights, data, title):
    boards, rows = '', []
    predictions = predict(weights, data)
    correct = 0
    for index, ((image, label), prediction) in enumerate(zip(data, predictions)):
        result = forward(weights, image)
        correct += int(label == prediction)
        boards += grid(f'{title} {index+1}／正解{CLASS_NAMES[label]}', image)
        rows.append((index+1, CLASS_NAMES[label], CLASS_NAMES[prediction], result['probabilities'][label], '正解' if label == prediction else '不正解'))
    show('<div class="boards">' + boards + '</div>' + table(f'{title}：{correct}/{len(data)}正解', ['画像','正解','予測','正解クラスへの出力','正誤'], rows))


In [ ]:
def loss_chart(history, step):
    # 座標を表示へ変換するだけ。historyは実際の全更新の損失。
    maximum = max(history) * 1.1
    x = lambda i: 65 + i / (len(history)-1) * 540
    y = lambda value: 250 - value / maximum * 210
    points = ' '.join(f'{x(i):.2f},{y(v):.2f}' for i,v in enumerate(history[:step+1]))
    marks = ''
    for i in (0, 1, 10, 50, 100):
        marks += f'<text x="{x(i)}" y="{292 if i == 1 else 275}">{i}</text>'
    for value in (0, maximum/2, maximum):
        marks += f'<path d="M65 {y(value)} H605" stroke="#aab6c3"/><text x="0" y="{y(value)+5}">{value:.3f}</text>'
    return f'<svg viewBox="0 0 700 340" role="img" aria-label="学習用の平均損失と更新回数"><text x="65" y="20">学習用の平均損失／現在{step}回：{history[step]:.6f}</text>{marks}<polyline points="{points}" fill="none" stroke="#155e75" stroke-width="3"/><circle cx="{x(step)}" cy="{y(history[step])}" r="6" fill="#a52b19"/><text x="260" y="325">重みの更新回数</text></svg>'


In [ ]:
def checkpoint_view(run, step):
    snapshot = run['snapshots'][step]
    weights = snapshot['weights']
    show(loss_chart(run['history'], step))
    for k in range(2):
        initial = [INITIAL_WEIGHTS[k*4:k*4+2], INITIAL_WEIGHTS[k*4+2:k*4+4]]
        current = [weights[k*4:k*4+2], weights[k*4+2:k*4+4]]
        show('<div class="boards">' + grid('初期フィルター'+'ab'[k], initial) + grid(f'{step}回のフィルター'+'ab'[k], current) + '</div>')
    sample_view(weights, samples([1,2]), '学習用：中央の位置1・2')
    sample_view(weights, samples([0,3]), '確認用：端の位置0・3（更新に不使用）')


## 1. 入力画像をリストとして読む

今いる段階：入力。黒い線のある画素が1、背景が0です。まず行0・列1の値を予想します。

**進め方：** 説明→定義→実行の順に、セル左の▶またはShift+Enterで一つずつ実行します。

**変更するもの：** 初回は変更しません。

**成功の目印：** image[0][1]が1。4行それぞれに4値があります。

**戻す：** この実行セルでIMAGEのコピーを作り直します。


In [ ]:
def matrix(a):
    if not a or not a[0] or any(len(row) != len(a[0]) for row in a):
        raise ValueError('空でない長方形の配列にしてください')
    if any(not math.isfinite(v) for row in a for v in row):
        raise ValueError('値は有限の数にしてください')
    return len(a), len(a[0])


In [ ]:
image = deepcopy(IMAGE)
print('行0:', image[0])
print('行0・列1:', image[0][1])
show(grid('入力画像：列1の縦線', image))


## 2. 一つの窓：4組の掛け算から出力1マスへ

今いる段階：窓と積和。窓は今見る範囲、フィルターは掛ける重みです。下の関数は積を順に保存し、その同じ積をconvolveが合計します。

**進め方：** 説明→定義→実行の順に、セル左の▶またはShift+Enterで一つずつ実行します。

**変更するもの：** window_positionを0→1にします。画像と重みは固定です。

**成功の目印：** 左上は0×1＋1×(−1)＋0×1＋1×(−1)=−2。右隣は＋2。

**戻す：** window_position=0にして実行します。


In [ ]:
def window_terms(image, kernel, top, left, padding=0):
    """一つの窓。top/leftは余白を含む画像上の開始位置。"""
    height, width = len(image), len(image[0])
    terms = []
    for u, kernel_row in enumerate(kernel):
        for v, weight in enumerate(kernel_row):
            row = top + u - padding
            col = left + v - padding
            inside = 0 <= row < height and 0 <= col < width
            pixel = image[row][col] if inside else 0
            terms.append({'row': row, 'col': col, 'added_zero': not inside,
                          'pixel': pixel, 'weight': weight, 'product': weight * pixel})
    return terms


In [ ]:
terms = window_terms(IMAGE, FILTER, top=0, left=0)
products = [term['product'] for term in terms]
print('4つの積:', products)
print('合計:', sum(products))


In [ ]:
def convolve(image, kernel, stride=1, padding=0, *, record=None):
    """フィルターを反転しない積和。recordへ各出力の計算を渡せる。"""
    height, width = matrix(image)
    kh, kw = matrix(kernel)
    if not isinstance(stride, int) or stride < 1 or not isinstance(padding, int) or padding < 0:
        raise ValueError('strideは正の整数、paddingは非負整数')
    if height + 2 * padding < kh or width + 2 * padding < kw:
        raise ValueError('フィルターが入力より大きすぎます')
    row_starts = range(0, height + 2 * padding - kh + 1, stride)
    col_starts = range(0, width + 2 * padding - kw + 1, stride)
    output = []
    for out_row, top in enumerate(row_starts):
        output_row = []
        for out_col, left in enumerate(col_starts):
            terms = window_terms(image, kernel, top, left, padding)
            products = [term['product'] for term in terms]
            total = sum(products)
            output_row.append(total)
            if record is not None:
                record({'output': [out_row, out_col], 'start': [top, left],
                        'terms': terms, 'total': total})
        output.append(output_row)
    return output


In [ ]:
window_position = 0
window_records = window_view(IMAGE, FILTER, position=window_position)
show(table('9か所の窓と出力の対応', ['窓の開始', '出力の行列', '合計'], [(str(r['start']), str(r['output']), r['total']) for r in window_records]))


**説明する問い：** 窓を右へ動かしたとき、変わったのは画像の値・見る範囲・重みのどれですか。これは重みの学習でしょうか。


## 3. 出力サイズ：窓の開始位置を数える

今いる段階：出力サイズ。追加0は破線で示します。開始位置は拡張後の画像で数え、計算表の座標は元画像で数えます。例えば余白1の開始0は元画像の−1に対応します。

**進め方：** 説明→定義→実行の順に、セル左の▶またはShift+Enterで一つずつ実行します。

**変更するもの：** shape_stride=1/2、shape_padding=0/1、shape_positionを選びます。条件を変えたらposition=0へ戻します。

**成功の目印：** 条件(1,0)→3×3、(1,1)→5×5、(2,0)→2×2、(2,1)→3×3。

**戻す：** stride=1、padding=0、position=0へ戻して実行します。


In [ ]:
shape_stride = 1
shape_padding = 0
shape_position = 0
shape_records = window_view(IMAGE, FILTER, shape_stride, shape_padding, shape_position)
starts = list(range(0, 4 + 2*shape_padding - 2 + 1, shape_stride))
print('縦・横それぞれの開始位置:', starts)
print('開始位置の個数:', len(starts), '→出力は', len(starts), '行×', len(starts), '列')


開始位置が分かってから式を読みます。`4 + 2*p`は余白込みの長さ、そこから窓の幅2を引くと動ける距離。移動幅sで割り、はみ出す端数を`//`で切り捨て、開始位置0の分を最後の＋1で数えます。


In [ ]:
shape_rows = []
for stride in (1, 2):
    for padding in (0, 1):
        starts = list(range(0, 4 + 2*padding - 2 + 1, stride))
        count = (4 + 2*padding - 2) // stride + 1
        shape_rows.append((stride, padding, str(starts), f'{count}×{count}'))
show(table('4条件の比較', ['移動幅','余白','開始位置（縦横共通）','出力サイズ'], shape_rows))


## 4. ReLUは値を変え、集約は4値を1個にする

今いる段階：ReLU→集約。ReLUでは形を変えません。プーリングは2×2の範囲を一つの値へまとめます。

**進め方：** 説明→定義→実行の順に、セル左の▶またはShift+Enterで一つずつ実行します。

**変更するもの：** pool_modeをmax/mean、pool_positionを0〜3、line_positionを0〜3にします。

**成功の目印：** 列1ならReLUの各行[0,2,0]。窓内[0,2,0,2]は最大2、平均1。

**戻す：** line_position=1、pool_mode="max"、pool_position=0へ戻します。


In [ ]:
def relu(image):
    """位置を変えず、負の値だけ0にする。"""
    matrix(image)
    output = []
    for row in image:
        output_row = []
        for value in row:
            output_row.append(max(0., value))
        output.append(output_row)
    return output


In [ ]:
def pool(image, mode='max', size=2, stride=1, *, record=None):
    height, width = matrix(image)
    if mode not in ('max', 'mean') or size < 1 or stride < 1 or min(height, width) < size:
        raise ValueError('プーリング条件を確認してください')
    output = []
    for out_row, top in enumerate(range(0, height - size + 1, stride)):
        output_row = []
        for out_col, left in enumerate(range(0, width - size + 1, stride)):
            values = []
            for u in range(size):
                for v in range(size):
                    values.append(image[top + u][left + v])
            total = max(values) if mode == 'max' else sum(values) / len(values)
            output_row.append(total)
            if record is not None:
                record({'output': [out_row, out_col], 'start': [top, left],
                        'values': values, 'mode': mode, 'total': total})
        output.append(output_row)
    return output


In [ ]:
def samples(positions):
    """0=縦線、1=横線。縦→横、それぞれ位置順。位置は0始まり。"""
    if any(position not in range(4) for position in positions):
        raise ValueError('位置は0〜3')
    data = []
    for label in range(2):
        for position in positions:
            image = []
            for row in range(4):
                pixels = []
                for col in range(4):
                    coordinate = col if label == 0 else row
                    pixels.append(float(coordinate == position))
                image.append(pixels)
            data.append((image, label))
    return data


In [ ]:
line_position = 1
pool_mode = 'max'
pool_position = 0
line_image = samples([line_position])[0][0]
raw = convolve(line_image, FILTER)
active = relu(raw)
show('<div class="boards">' + grid('画像', line_image) + grid('畳み込み', raw) + grid('ReLU後', active) + '</div>')
pooling_view(active, pool_mode, pool_position)


In [ ]:
comparison = []
for position in range(4):
    active_line = relu(convolve(samples([position])[0][0], FILTER))
    for mode in ('max', 'mean'):
        comparison.append((position, mode, str(pool(active_line, mode))))
show(table('線の位置×集約方法：基準の列1と比べる', ['列','方法','2×2の結果'], comparison))


**失われる情報：** [0,2,0,2]も[2,0,2,0]も最大2・平均1ですが配置は異なります。最大2だけでは[2,0,0,0]とも区別できません。列1→列2では集約結果も変わります。「必ず位置ずれに強くなる」とは言えません。


## 5. 分類：2枚の特徴マップから二つのスコアへ

今いる段階：分類。この節は学習前の固定初期重みを使います。手作りFILTERとは異なり、各フィルターにバイアスも足します。

**進め方：** 説明→定義→実行の順に、セル左の▶またはShift+Enterで一つずつ実行します。

**変更するもの：** sample_indexを0〜7にします。縦線の列0〜3、横線の行0〜3の順です。

**成功の目印：** 2枚の2×2をa1〜a4、b1〜b4へ並べ、8値×クラス別重み＋バイアスで2スコアができます。

**戻す：** sample_index=1に戻して実行。学習節を実行してもこの節はINITIAL_WEIGHTSを使います。


In [ ]:
def score_terms(weights, features, label):
    """8値とクラス別の8重みを対応させる。forwardもこの積を使う。"""
    terms = []
    for index, feature in enumerate(features):
        weight = weights[10 + label * 8 + index]
        name = ('a' if index < 4 else 'b') + str(index % 4 + 1)
        terms.append({'name': name, 'feature': feature,
                      'weight': weight, 'product': weight * feature})
    return terms


In [ ]:
def forward(weights, image):
    if len(weights) != 28 or matrix(image) != (4, 4):
        raise ValueError('学習モデルは28パラメーター・4×4入力です')
    maps, activated, pooled, features = [], [], [], []
    for k in range(2):
        kernel = [weights[k * 4:k * 4 + 2], weights[k * 4 + 2:k * 4 + 4]]
        convolution = convolve(image, kernel)
        raw = []
        for row in convolution:
            raw.append([value + weights[8 + k] for value in row])
        act = relu(raw)
        small = pool(act)
        maps.append(raw)
        activated.append(act)
        pooled.append(small)
        for row in small:
            features.extend(row)  # aの左上→右上→左下→右下、その後b。
    logits = []
    for label in range(2):
        terms = score_terms(weights, features, label)
        products = [term['product'] for term in terms]
        logits.append(sum(products) + weights[26 + label])
    maximum = max(logits)
    exps = [math.exp(value - maximum) for value in logits]
    denominator = sum(exps)
    logsum = maximum + math.log(denominator)
    return {'maps': maps, 'relu': activated, 'pooled': pooled, 'features': features,
            'logits': logits, 'probabilities': [value / denominator for value in exps],
            'log_probabilities': [value - logsum for value in logits]}


In [ ]:
def predict(weights, data):
    predictions = []
    for image, _ in data:
        probabilities = forward(weights, image)['probabilities']
        # 同値なら先にある0（縦線）を選ぶ。
        label = max(range(2), key=probabilities.__getitem__)
        predictions.append(label)
    return predictions


In [ ]:
sample_index = 1
classification_data = samples(range(4))
class_image, correct_label = classification_data[sample_index]
classification = forward(INITIAL_WEIGHTS, class_image)
print('8値:', classification['features'])
print('2スコア:', classification['logits'])
classification_view(INITIAL_WEIGHTS, class_image, correct_label)


softmaxは各スコアから最大値を引いてexpを計算し、2個のexpの合計で割ります。大きな指数による桁あふれを避けても結果の比率は同じです。二つの出力の合計は1ですが、正解を保証する値ではありません。予測と正解を分けて読んでください。


In [ ]:
scores = classification['logits']
shifted = [score - max(scores) for score in scores]
exp_values = [math.exp(value) for value in shifted]
show(table('softmaxの内訳', ['クラス','スコア','最大値を引く','exp','合計で割る'], [(CLASS_NAMES[i],scores[i],shifted[i],exp_values[i],classification['probabilities'][i]) for i in range(2)]))


## 6. 学習：予測→正解との比較→損失→重み更新

今いる段階：学習。学習用は中央の位置1・2、確認用は端の位置0・3。確認用の4画像は更新に使いません。平均損失は正解クラスへの出力から求め、正解数とは別の指標です。

**進め方：** 説明→定義→実行の順に、セル左の▶またはShift+Enterで一つずつ実行します。

**変更するもの：** まず変更せず、初期状態の8画像を見ます。

**成功の目印：** 初期予測は学習用[横,横,横,縦]。学習用平均損失は約0.765126。

**戻す：** このセルを再実行すると初期重みで表示し直します。


In [ ]:
def loss(weights, data):
    individual_losses = []
    for image, label in data:
        result = forward(weights, image)
        individual_losses.append(-result['log_probabilities'][label])
    return sum(individual_losses) / len(data)


In [ ]:
training_data = samples([1, 2])
checking_data = samples([0, 3])
sample_view(INITIAL_WEIGHTS, training_data, '学習用：位置1・2')
sample_view(INITIAL_WEIGHTS, checking_data, '確認用：位置0・3')
loss_rows = []
for index, (image, label) in enumerate(training_data):
    result = forward(INITIAL_WEIGHTS, image)
    loss_rows.append((index+1, CLASS_NAMES[label], result['probabilities'][label], -result['log_probabilities'][label]))
show(table('1画像の損失=−log(正解クラスへの出力)', ['画像','正解','正解クラスへの出力','損失'], loss_rows))
print('4画像の損失の合計÷4 =', loss(INITIAL_WEIGHTS, training_data))


### 6a. 一つの重みで更新の意味を確かめる

重みをほんの少し増やす場合と減らす場合の損失を比べます。差÷(2×1e-6)が数値微分による勾配です。勾配が正なら重みを減らす方向へ進みます。28個すべてを同じ更新前の重みで調べてから一括更新します。ReLUの0や最大値の同値付近では近似であり、大規模CNNに向く計算法ではありません。


In [ ]:
def gradient_term(weights, data, index):
    """一つの重みを±1e-6だけ試す。元の重みは変えない。"""
    plus, minus = list(weights), list(weights)
    plus[index] += 1e-6
    minus[index] -= 1e-6
    loss_plus = loss(plus, data)
    loss_minus = loss(minus, data)
    gradient = (loss_plus - loss_minus) / 2e-6
    return {'index': index, 'weight': weights[index], 'loss_plus': loss_plus,
            'loss_minus': loss_minus, 'gradient': gradient}


In [ ]:
weight_index = 0
learning_rate = 0.1
update_term = gradient_term(INITIAL_WEIGHTS, training_data, weight_index)
new_weight = update_term['weight'] - learning_rate * update_term['gradient']
show(table('フィルターaの左上重みを1回更新する', ['現在の重み','+1e-6の損失','−1e-6の損失','勾配','学習率','更新後'], [(update_term['weight'],update_term['loss_plus'],update_term['loss_minus'],update_term['gradient'],learning_rate,new_weight)]))
print('微小な損失差は表示で消えることがあるので内部値も確認:', update_term)


### 6b. 同じ初期値から100回だけ学習する

`record`は途中の値を保存するための引数です。更新方法は変えません。0・1・10・50・100回を保存し、途中の結果を補間して作りません。次の実行セルは学習率0と0.1を比較するため各1回、100更新を実行します。実行が終わるまで待ってください。


In [ ]:
def train(rate=.1, steps=100, initial=INITIAL_WEIGHTS, *, record=None):
    """同じ初期値のコピーから全件一括更新。record(step, weights, loss)。"""
    if not math.isfinite(rate) or rate < 0 or not isinstance(steps, int) or not 0 <= steps <= 1000:
        raise ValueError('rateは非負の有限値、stepsは0〜1000の整数')
    weights = list(initial)
    data = samples([1, 2])
    history = [loss(weights, data)]
    if record is not None:
        record(0, weights.copy(), history[0])
    for step in range(1, steps + 1):
        gradients = []
        for index in range(28):
            term = gradient_term(weights, data, index)
            gradients.append(term['gradient'])
        # 全28個の勾配を同じ更新前の重みから求めてから、一括で更新。
        updated = []
        for value, gradient in zip(weights, gradients):
            updated.append(value - rate * gradient)
        weights = updated
        history.append(loss(weights, data))
        if record is not None:
            record(step, weights.copy(), history[-1])
    return {'weights': weights, 'history': history,
            'train_predictions': predict(weights, data),
            'heldout_predictions': predict(weights, samples([0, 3]))}


In [ ]:
def training_run(rate=.1):
    """100回を一度実行し、指定した5時点を保存する。補間はしない。"""
    snapshots = {}
    def record(step, weights, value):
        if step in (0, 1, 10, 50, 100):
            snapshots[step] = {'weights': weights, 'loss': value}
    result = train(rate=rate, record=record)
    return {**result, 'snapshots': snapshots}


In [ ]:
learning_runs = {}
for rate in (0, 0.1):
    print('初期値から100回を計算中：学習率', rate)
    learning_runs[rate] = training_run(rate)
print('完了。以下の表示セルは再学習しません。')


### 6c. 保存した状態を選んで見返す

`selected_rate`は0／0.1、`checkpoint`は0／1／10／50／100。このセルは保存済みの値を表示するだけです。戻すにはcheckpoint=0。学習を最初からやり直す場合だけ6bの実行セルへ戻ります。


In [ ]:
selected_rate = 0.1
checkpoint = 100
selected_run = learning_runs[selected_rate]
checkpoint_view(selected_run, checkpoint)


In [ ]:
summary_rows = []
for rate, run in learning_runs.items():
    for step, state in run['snapshots'].items():
        train_predictions = predict(state['weights'], training_data)
        held_predictions = predict(state['weights'], checking_data)
        train_correct = sum(p == label for p, (_,label) in zip(train_predictions, training_data))
        held_correct = sum(p == label for p, (_,label) in zip(held_predictions, checking_data))
        summary_rows.append((rate,step,state['loss'],f'{train_correct}/4',f'{held_correct}/4'))
show(table('学習率と途中状態：平均損失と正解数を分ける', ['学習率','回数','学習用平均損失','学習用正解数','確認用正解数'], summary_rows))


**結果の説明：** 学習率0なら重みも損失も変わりません。0.1の100回後は学習用4/4、確認用2/4です。確認用の端にある縦線2枚を横線と誤認します。学習用の成功だけで、別位置や実写真にも成功するとは言えません。この結果だけから失敗原因を一つに断定しないでください。


## 7. 基準結果を保存する

以下は既存の検証用出力です。読み方はここまでの図表で確認済みです。baseline_resultは実験で上書きしません。demoは同じ初期値から基準の100回を計算します。


In [ ]:
def rounded(value):
    if isinstance(value, float):
        return round(value, 6)
    if isinstance(value, (tuple, list)):
        return [rounded(v) for v in value]
    if isinstance(value, dict):
        return {k: rounded(v) for k, v in value.items()}
    return value


In [ ]:
def demo():
    raw = convolve(IMAGE, FILTER)
    fit = train()
    return rounded({'convolution': raw, 'relu': relu(raw),
                    'stride2': convolve(IMAGE, FILTER, stride=2),
                    'padding_shape': [len(convolve(IMAGE, FILTER, padding=1))] * 2,
                    'max_pool': pool(relu(raw)), 'mean_pool': pool(relu(raw), 'mean'),
                    'parameter_count': len(INITIAL_WEIGHTS),
                    'loss_before': fit['history'][0], 'loss_after': fit['history'][-1],
                    'train_before': predict(INITIAL_WEIGHTS, samples([1, 2])),
                    'train_after': fit['train_predictions'], 'heldout_after': fit['heldout_predictions'],
                    'labels': [0, 0, 1, 1]})


In [ ]:
baseline_result = demo()
print(json.dumps(baseline_result, ensure_ascii=False, indent=2))
baseline_fit = learning_runs[0.1]


## 実験A：フィルターの符号だけ変える

予想：正負が逆になった後、ReLUでは何が残る？ sign=-1で1行目[2,-2,0]。戻す：sign=1。

実行前の予想を書き、実行後に変更前との違いを説明します。元データは書き換えません。


In [ ]:
sign = -1
trial_image = deepcopy(IMAGE)
trial_filter = [[sign*v for v in row] for row in deepcopy(FILTER)]
window_view(trial_image, trial_filter)


## 実験B：移動幅だけ変える

予想：どの開始位置を使わなくなる？ trial_stride=2で2×2・各行[-2,0]。戻す：trial_stride=1。

実行前の予想を書き、実行後に変更前との違いを説明します。元データは書き換えません。


In [ ]:
trial_stride = 2
window_view(deepcopy(IMAGE), deepcopy(FILTER), stride=trial_stride)


## 実験C：最大から平均へ変える

予想：[0,2,0,2]を平均すると？ trial_mode=meanで各マス1。戻す：trial_mode=max。

実行前の予想を書き、実行後に変更前との違いを説明します。元データは書き換えません。


In [ ]:
trial_mode = 'mean'
trial_features = relu(convolve(deepcopy(IMAGE), deepcopy(FILTER)))
pooling_view(trial_features, trial_mode)


## 実験D：学習率0と0.1を比べる

予想：重みが変わらない場合、損失はどうなる？ このセルは同じ初期値からそれぞれ100回を実計算します。前の学習を引き継ぎません。戻す：セルを再実行。

実行前の予想を書き、実行後に変更前との違いを説明します。元データは書き換えません。


In [ ]:
for trial_rate in (0, 0.1):
    trial_fit = train(rate=trial_rate, initial=list(INITIAL_WEIGHTS))
    print('学習率',trial_rate,'最初の損失',trial_fit['history'][0],'最後の損失',trial_fit['history'][-1])
    sample_view(trial_fit['weights'], samples([0,3]), '確認用')


## 振り返りと復帰

1. 入力画像とフィルターを区別し、出力の左上が−2になる理由を説明できますか。
2. 窓の移動、ReLU、集約、学習は、それぞれ何を変えましたか。
3. 分類の8値はどのマスから来ましたか。予測と正解を別々に言えますか。
4. 学習用4/4と確認用2/4から、どこまで言えますか。

実行順が分からない／NameError：カーネル再起動→すべて実行。位置の範囲エラー：節に書いた初期値へ戻します。依存不足：既存venvとカーネルが一致するか確認。編集中は別名保存し、元ファイルは残します。途中で停止したら再起動して最初から実行してください。

Python版も標準ライブラリだけで実行できます。`python cnn_basics.py --explain convolution`で同じ窓の計算表、`--explain size`、`activation`、`classification`、`learning`、`all`で各テーマを表示します。既存の`--demo --json`は基準結果の照合用です。
